# 06 — Entraînement RoBERTa-large NER (**6 labels**)

## Pipeline hybride (après entraînement)

| Entité | Outil |
|--------|-------|
| SKILL, JOB_TITLE, COMPANY, DATE, DEGREE, INSTITUTION | **RoBERTa-large** (ce notebook) |
| EXPERIENCE_DESC | **LLM Groq** (intégration app plus tard) |
| CERTIFICATION | **Retirée** |

## Prérequis
1. Notebook **03** exécuté → `bloc3_balanced_train/dev.spacy`
2. Exports Doccano uploadés : **`TRAIN.jsonl`** + **`DEV.jsonl`** dans `bloc1_processed/`
3. GPU Colab (A100 recommandé)

## Ordre après correction Doccano
1. **03** — Run all (détecte TRAIN.jsonl + DEV.jsonl)
2. **06** — cellules 1→7 ci-dessous (**sans** audit Doccano)

## Objectif
**F1 dev ≥ 0.85** sur les **6 labels** (sans CERTIFICATION ni EXPERIENCE_DESC).

## Sortie
`MyDrive/cvextraction/Datasets/models/cv_ner_roberta_best/`


In [2]:
#@title Configuration Colab + GPU A100 (SAFE VERSION)

from google.colab import drive
import os, json, shutil, subprocess, sys
from pathlib import Path
from collections import Counter

drive.mount('/content/drive')

# ============================================================
# PATHS
# ============================================================

PROJECT    = '/content/drive/MyDrive/cvextraction'
BASE       = os.path.join(PROJECT, 'Datasets')

BLOC3      = os.path.join(BASE, 'bloc3_processed')

# ⚠️ IMPORTANT: utilise tes vrais fichiers générés
TRAIN_PATH = os.path.join(BLOC3, 'bloc3_balanced_train.spacy')
DEV_PATH   = os.path.join(BLOC3, 'bloc3_balanced_dev.spacy')

MODEL_DIR  = os.path.join(BASE, 'models')
TRAIN_OUT  = os.path.join(MODEL_DIR, 'cv_ner_trf_run1')
MODEL_BEST = os.path.join(MODEL_DIR, 'cv_ner_trf_best1')
CONFIG_DIR = os.path.join(MODEL_DIR, 'configs')
CONFIG_PATH = os.path.join(CONFIG_DIR, 'ner_transformer.cfg')

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(CONFIG_DIR, exist_ok=True)

# ============================================================
# CHECK FILES
# ============================================================

missing = []
for p in [TRAIN_PATH, DEV_PATH]:
    if not os.path.isfile(p):
        missing.append(p)

if missing:
    print("❌ Fichiers manquants :")
    for m in missing:
        print(" -", m)

    raise FileNotFoundError(
        "Tu dois d'abord exécuter la cellule DocBin (conversion spaCy)"
    )

# ============================================================
# SPACY + GPU
# ============================================================

import spacy
from spacy.tokens import DocBin
from spacy.training import Example
from spacy.util import filter_spans

try:
    from spacy import prefer_gpu
    prefer_gpu()
    print("GPU spaCy activé")
except Exception as e:
    print("GPU non activé :", e)

print("\nTrain :", TRAIN_PATH)
print("Dev   :", DEV_PATH)
print("Out   :", MODEL_BEST)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU spaCy activé

Train : /content/drive/MyDrive/cvextraction/Datasets/bloc3_processed/bloc3_balanced_train.spacy
Dev   : /content/drive/MyDrive/cvextraction/Datasets/bloc3_processed/bloc3_balanced_dev.spacy
Out   : /content/drive/MyDrive/cvextraction/Datasets/models/cv_ner_trf_best1


In [ ]:
#@title Setup spaCy Transformer PROPRE (A100)

from google.colab import drive
drive.mount('/content/drive')

# =========================================================
# INSTALL CLEAN
# =========================================================

!pip uninstall -y numpy cupy-cuda12x spacy thinc blis cymem murmurhash preshed
!pip install -q numpy==1.26.4
!pip install -q "spacy[cuda12x]==3.7.5"
!pip install -q spacy-transformers==1.3.5
!pip install -q spacy-lookups-data

# IMPORTANT
import os
os.kill(os.getpid(), 9)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
Found existing installation: cupy-cuda12x 14.0.1
Uninstalling cupy-cuda12x-14.0.1:
  Successfully uninstalled cupy-cuda12x-14.0.1
Found existing installation: spacy 3.8.14
Uninstalling spacy-3.8.14:
  Successfully uninstalled spacy-3.8.14
Found existing installation: thinc 8.3.13
Uninstalling thinc-8.3.13:
  Successfully uninstalled thinc-8.3.13
Found existing installation: blis 1.3.3
Uninstalling blis-1.3.3:
  Successfully uninstalled blis-1.3.3
Found existing installation: cymem 2.0.13
Uninstalling cymem-2.0.13:
  Successfully uninstalled cymem-2.0.13
Found existing installation: murmurhash 1.0.15
Uninstalling murmurhash-1.0.15:
  Successfully uninstalled murmurhash-1.0.15
Found existing installation: preshed 3.0.13
Uninstalling preshed-3.0.13:
  Succes

In [1]:
#@title 🚀 RoBERTa-large — 6 labels NER

LABELS = [
    "SKILL", "JOB_TITLE", "COMPANY", "DATE", "DEGREE", "INSTITUTION",
]

TRANSFORMER_MODEL = "roberta-large"
MAX_EPOCHS      = 50
PATIENCE        = 2000
EVAL_FREQUENCY  = 200
LEARN_RATE      = 2e-5
WARMUP_STEPS    = 500
DROPOUT         = 0.10
BATCH_SIZE      = 16
GRAD_ACCUMULATE = 4
GPU_ID          = 0
SEED            = 42
CLEAN_SPANS     = True
TARGET_F1       = 0.85

print("=" * 60)
print("RoBERTa-large | 6 labels | cible F1 >=", TARGET_F1)
print("=" * 60)
for k, v in [
    ("Transformer", TRANSFORMER_MODEL),
    ("Max epochs", MAX_EPOCHS),
    ("Patience", PATIENCE),
    ("LR", LEARN_RATE),
    ("Batch x accum", f"{BATCH_SIZE} x {GRAD_ACCUMULATE}"),
]:
    print(f"  {k:16s} : {v}")
print("=" * 60)


RoBERTa-large | 6 labels | cible F1 >= 0.85
  Transformer      : roberta-large
  Max epochs       : 50
  Patience         : 2000
  LR               : 2e-05
  Batch x accum    : 16 x 4


**Note :** hyperparametres dans la cellule **RoBERTa-large -- 6 labels NER** (cellule 3).\n\nOrdre : Setup spaCy -> Hyperparams -> Clean spans -> Config -> Train -> Eval\n

In [3]:
#@title Chargement + nettoyage spans NER (renforcé)

nlp_vocab = spacy.blank('en')
train_db = DocBin().from_disk(TRAIN_PATH)
dev_db   = DocBin().from_disk(DEV_PATH)

train_docs = list(train_db.get_docs(nlp_vocab.vocab))
dev_docs   = list(dev_db.get_docs(nlp_vocab.vocab))

LABEL_PRIORITY = {
    "EXPERIENCE_DESC": 8, "JOB_TITLE": 7, "COMPANY": 6,
    "DEGREE": 5, "INSTITUTION": 5, "CERTIFICATION": 5,
    "DATE": 4, "SKILL": 3,
}

def count_entities(docs):
    c = Counter()
    for d in docs:
        for ent in d.ents:
            c[ent.label_] += 1
    return c

def resolve_doc_overlaps(spans):
    ordered = sorted(
        spans,
        key=lambda s: (-(LABEL_PRIORITY.get(s.label_, 0)), -(s.end_char - s.start_char), s.start_char),
    )
    kept, occupied = [], []
    for sp in ordered:
        if any(not (sp.end_char <= a or sp.start_char >= b) for a, b in occupied):
            continue
        kept.append(sp)
        occupied.append((sp.start_char, sp.end_char))
    return sorted(kept, key=lambda s: s.start_char)

def clean_docs(docs):
    cleaned, removed = [], 0
    for doc in docs:
        valid_spans = []
        for ent in doc.ents:
            stripped = doc.text[ent.start_char:ent.end_char].strip()
            if not stripped or len(stripped) < 2:
                removed += 1
                continue
            start = doc.text.find(stripped, ent.start_char)
            if start == -1:
                removed += 1
                continue
            span = doc.char_span(start, start + len(stripped), label=ent.label_, alignment_mode='contract')
            if span is not None:
                valid_spans.append(span)
            else:
                removed += 1
        valid_spans = resolve_doc_overlaps(valid_spans)
        doc.ents = filter_spans(valid_spans)
        cleaned.append(doc)
    print(f'Spans supprimés / recalés : {removed}')
    return cleaned

print(f'Avant nettoyage — train: {len(train_docs)} | dev: {len(dev_docs)}')
print('Entités train :', dict(count_entities(train_docs)))

if CLEAN_SPANS:
    train_docs = clean_docs(train_docs)
    dev_docs   = clean_docs(dev_docs)
    print('Entités train (après) :', dict(count_entities(train_docs)))

TRAIN_CLEAN = os.path.join(BLOC3, 'bloc3_train_clean.spacy')
DEV_CLEAN   = os.path.join(BLOC3, 'bloc3_dev_clean.spacy')

def save_docbin(docs, path):
    db = DocBin(attrs=['ENT_IOB', 'ENT_TYPE'], store_user_data=False)
    for d in docs:
        db.add(d)
    db.to_disk(path)

save_docbin(train_docs, TRAIN_CLEAN)
save_docbin(dev_docs, DEV_CLEAN)
print('Corpus nettoyé :', TRAIN_CLEAN)


/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


Avant nettoyage — train: 1291 | dev: 60
Entités train : {'SKILL': 16544, 'COMPANY': 3262, 'JOB_TITLE': 3821, 'DATE': 4692, 'EXPERIENCE_DESC': 3218, 'DEGREE': 1284, 'INSTITUTION': 1280, 'CERTIFICATION': 666}
Spans supprimés / recalés : 6908
Spans supprimés / recalés : 271
Entités train (après) : {'SKILL': 11323, 'JOB_TITLE': 3755, 'DATE': 4471, 'EXPERIENCE_DESC': 3218, 'DEGREE': 1284, 'INSTITUTION': 1232, 'COMPANY': 1934, 'CERTIFICATION': 642}
Corpus nettoyé : /content/drive/MyDrive/cvextraction/Datasets/bloc3_processed/bloc3_train_clean.spacy


In [4]:
#@title ⚙️ CONFIG RoBERTa-large (6 labels)

import subprocess, sys, re
from pathlib import Path

CONFIG_PATH = os.path.join(BASE, "models", "configs", "ner_roberta_6l.cfg")
os.makedirs(os.path.dirname(CONFIG_PATH), exist_ok=True)

result = subprocess.run([
    sys.executable, "-m", "spacy", "init", "config", CONFIG_PATH,
    "--lang", "en", "--pipeline", "transformer,ner",
    "--optimize", "accuracy", "--gpu", "--force",
], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    raise RuntimeError(result.stderr)

cfg = Path(CONFIG_PATH).read_text(encoding="utf-8")
cfg = re.sub(r'name = "roberta-base"', f'name = "{TRANSFORMER_MODEL}"', cfg)
if f'name = "{TRANSFORMER_MODEL}"' not in cfg:
    cfg = re.sub(r'name = "roberta-large"', f'name = "{TRANSFORMER_MODEL}"', cfg)

for pat, repl in [
    (r"max_epochs = \d+", f"max_epochs = {MAX_EPOCHS}"),
    (r"dropout = [0-9.]+", f"dropout = {DROPOUT}"),
    (r"patience = \d+", f"patience = {PATIENCE}"),
    (r"eval_frequency = \d+", f"eval_frequency = {EVAL_FREQUENCY}"),
    (r"accumulate_gradient = \d+", f"accumulate_gradient = {GRAD_ACCUMULATE}"),
    (r"initial_rate = [0-9.e-]+", f"initial_rate = {LEARN_RATE}"),
    (r"warmup_steps = \d+", f"warmup_steps = {WARMUP_STEPS}"),
]:
    cfg = re.sub(pat, repl, cfg)
cfg = re.sub(r"(?m)^seed = \d+", f"seed = {SEED}", cfg, count=1)
cfg = re.sub(r"(?m)^batch_size = \d+", f"batch_size = {BATCH_SIZE}", cfg, count=1)
if "mixed_precision = false" in cfg:
    cfg = cfg.replace("mixed_precision = false", "mixed_precision = true")

Path(CONFIG_PATH).write_text(cfg, encoding="utf-8")
print("CONFIG :", CONFIG_PATH)


ℹ Generated config template specific for your use case
- Language: en
- Pipeline: ner
- Optimize for: accuracy
- Hardware: GPU
- Transformer: roberta-base
✔ Auto-filled config with all values
✔ Saved config
/content/drive/MyDrive/cvextraction/Datasets/models/configs/ner_roberta_6l.cfg
You can now add your data and train your pipeline:
python -m spacy train ner_roberta_6l.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy

CONFIG : /content/drive/MyDrive/cvextraction/Datasets/models/configs/ner_roberta_6l.cfg


In [5]:
#@title 🏋️ Entraînement RoBERTa-large → cv_ner_roberta_best

import os, sys, shutil, subprocess, torch, json
from pathlib import Path

TRAIN_OUT = os.path.join(BASE, "models", "cv_ner_trf_run_6l")
FINAL_MODEL = os.path.join(BASE, "models", "cv_ner_roberta_best")

if os.path.isdir(TRAIN_OUT):
    shutil.rmtree(TRAIN_OUT)
os.makedirs(TRAIN_OUT, exist_ok=True)

cmd = [
    sys.executable, "-m", "spacy", "train", CONFIG_PATH,
    "--output", TRAIN_OUT,
    "--paths.train", TRAIN_CLEAN,
    "--paths.dev", DEV_CLEAN,
    "--gpu-id", str(GPU_ID),
]
print("CUDA:", torch.cuda.is_available())
print(" ".join(cmd))

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    raise RuntimeError("Training failed")

best = os.path.join(TRAIN_OUT, "model-best")
if os.path.isdir(FINAL_MODEL):
    shutil.rmtree(FINAL_MODEL)
shutil.copytree(best, FINAL_MODEL)
meta = json.loads(Path(FINAL_MODEL, "meta.json").read_text(encoding="utf-8"))
f1 = meta.get("performance", {}).get("ents_f", 0)
print(f"\nmodel-best F1 (6 labels) : {f1:.4f} ({f1*100:.2f}%)")
print("Deploy:", FINAL_MODEL)


CUDA: True
/usr/bin/python3 -m spacy train /content/drive/MyDrive/cvextraction/Datasets/models/configs/ner_roberta_6l.cfg --output /content/drive/MyDrive/cvextraction/Datasets/models/cv_ner_trf_run_6l --paths.train /content/drive/MyDrive/cvextraction/Datasets/bloc3_processed/bloc3_train_clean.spacy --paths.dev /content/drive/MyDrive/cvextraction/Datasets/bloc3_processed/bloc3_dev_clean.spacy --gpu-id 0
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning:

_(Entraînement DeBERTa 2 phases remplacé par RoBERTa-large 6 labels ci-dessus.)_


## Phase 0 — Audit Doccano *(déjà fait — ne pas relancer)*

Tu as corrigé **DEV.jsonl** + **TRAIN.jsonl** → passe directement à l'entraînement.

Les cellules **Audit** et **Export JSONL Doccano** sont optionnelles (debug seulement).


In [ ]:
#@title 🔍 Audit Doccano — export 60 dev + 20 train

import csv
import hashlib
import json
import os
from pathlib import Path
from collections import Counter

import spacy
from spacy.util import filter_spans

# --- Chemins ---
if "BASE" not in globals():
    PROJECT = "/content/drive/MyDrive/cvextraction"
    BASE = os.path.join(PROJECT, "Datasets")
    BLOC3 = os.path.join(BASE, "bloc3_processed")

MODEL_DIR = os.path.join(BASE, "models")
TRAIN_OUT = os.path.join(MODEL_DIR, "cv_ner_trf_run_6l")

LABELS_6 = globals().get("LABELS") or [
    "SKILL", "JOB_TITLE", "COMPANY", "DATE", "DEGREE", "INSTITUTION",
]
TRAIN_REVIEW_N = 20
DEV_EXPECT = 60

AUDIT_DIR = os.path.join(BLOC3, "annotation_audit")
os.makedirs(AUDIT_DIR, exist_ok=True)

try:
    import spacy_transformers  # noqa: F401
except ImportError:
    import subprocess
    import sys
    print("Installation spacy-transformers...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "spacy-transformers==1.3.5",
    ])
    import spacy_transformers  # noqa: F401

DEV_JSON = os.path.join(BLOC3, "dev_cv.json")
TRAIN_JSON = os.path.join(BLOC3, "train_cv.json")

if not os.path.isfile(DEV_JSON):
    raise FileNotFoundError(
        f"{DEV_JSON} introuvable — exécute notebook 03 (cellule Split train/dev)."
    )

with open(DEV_JSON, encoding="utf-8") as f:
    dev_records = json.load(f)

train_records = []
if os.path.isfile(TRAIN_JSON):
    with open(TRAIN_JSON, encoding="utf-8") as f:
        train_records = json.load(f)
else:
    print("⚠️ train_cv.json absent — export dev seulement.")

print(f"Dev  : {len(dev_records)} CV")
print(f"Train: {len(train_records)} CV")
if len(dev_records) != DEV_EXPECT:
    print(f"⚠️ Attendu {DEV_EXPECT} dev, trouvé {len(dev_records)}")


def doc_fingerprint(text):
    norm = " ".join(str(text).split()).lower()
    return hashlib.sha256(norm.encode("utf-8")).hexdigest()


def cv_to_spacy_doc(nlp_blank, cv):
    doc = nlp_blank.make_doc(cv["text"])
    spans = []
    for start, end, label in cv.get("entities", []):
        if label not in LABELS_6:
            continue
        span = doc.char_span(int(start), int(end), label=label, alignment_mode="contract")
        if span is None:
            span = doc.char_span(int(start), int(end), label=label, alignment_mode="expand")
        if span is not None:
            spans.append(span)
    doc.ents = filter_spans(spans)
    return doc


def span_snippet(text, start, end, pad=40):
    a = max(0, start - pad)
    b = min(len(text), end + pad)
    return text[a:b].replace("\n", " ")


def model_is_complete(path):
    tr = Path(path) / "transformer" / "model"
    return tr.is_file() and tr.stat().st_size > 50_000_000


def find_best_model(model_dir, train_out):
    for c in [
        os.path.join(train_out, "model-last"),
        os.path.join(train_out, "model-best"),
        os.path.join(model_dir, "cv_ner_roberta_best"),
    ]:
        if model_is_complete(c):
            tr = Path(c) / "transformer" / "model"
            mb = tr.stat().st_size // 1024 // 1024
            print(f"Modèle OK : {c} ({mb} Mo)")
            return c
    raise FileNotFoundError("transformer/model manquant — cellule Réparer copie modèle.")


def audit_cv_list(nlp, cv_list, split_name):
    rows, detail, label_errors = [], [], Counter()
    nlp_blank = spacy.blank("en")

    for idx, cv in enumerate(cv_list):
        gold = cv_to_spacy_doc(nlp_blank, cv)
        pred = nlp(gold.text)
        gold_e = {(e.start_char, e.end_char, e.label_) for e in gold.ents}
        pred_e = {(e.start_char, e.end_char, e.label_) for e in pred.ents if e.label_ in LABELS_6}
        fn, fp = gold_e - pred_e, pred_e - gold_e
        tp = len(gold_e & pred_e)
        n_err = len(fn) + len(fp)
        err_by_label = Counter(l for _, _, l in list(fn) + list(fp))
        for lbl, n in err_by_label.items():
            label_errors[lbl] += n

        p = tp / (tp + len(fp)) if (tp + len(fp)) else (1.0 if not gold_e else 0.0)
        r = tp / (tp + len(fn)) if (tp + len(fn)) else (1.0 if not pred_e else 0.0)
        f1_doc = (2 * p * r / (p + r)) if (p + r) else 1.0

        hint = cv["text"][:160].replace("\n", " ").strip()
        rows.append({
            "split": split_name,
            "idx": idx,
            "fingerprint": cv.get("fingerprint") or doc_fingerprint(cv["text"]),
            "search_hint": hint,
            "n_gold": len(gold_e),
            "n_errors": n_err,
            "n_fn": len(fn),
            "n_fp": len(fp),
            "f1_doc": round(f1_doc, 4),
            "err_SKILL": err_by_label.get("SKILL", 0),
            "err_INSTITUTION": err_by_label.get("INSTITUTION", 0),
            "err_COMPANY": err_by_label.get("COMPANY", 0),
            "err_DEGREE": err_by_label.get("DEGREE", 0),
            "err_JOB_TITLE": err_by_label.get("JOB_TITLE", 0),
            "err_DATE": err_by_label.get("DATE", 0),
            "_cv": cv,
            "_fn": fn,
            "_fp": fp,
        })

        for s, e, lbl in fn:
            detail.append({"split": split_name, "idx": idx, "type": "FN", "label": lbl,
                           "snippet": span_snippet(cv["text"], s, e), "search_hint": hint})
        for s, e, lbl in fp:
            detail.append({"split": split_name, "idx": idx, "type": "FP", "label": lbl,
                           "snippet": span_snippet(cv["text"], s, e), "search_hint": hint})

    return rows, detail, label_errors


model_path = find_best_model(MODEL_DIR, TRAIN_OUT)
nlp_audit = spacy.load(model_path)

print("\nAudit dev (60 CV)...")
dev_rows, dev_detail, dev_label_errors = audit_cv_list(nlp_audit, dev_records, "dev")
dev_rows.sort(key=lambda x: (-x["n_errors"], -x["err_COMPANY"] - x["err_INSTITUTION"] - x["err_SKILL"], x["idx"]))
for i, row in enumerate(dev_rows, start=1):
    row["priority"] = i

train_rows, train_detail = [], []
if train_records:
    print("Audit train → top 20 erreurs...")
    all_train, _, _ = audit_cv_list(nlp_audit, train_records, "train")
    all_train.sort(key=lambda x: (-x["n_errors"], -x["err_COMPANY"] - x["err_INSTITUTION"] - x["err_SKILL"], x["idx"]))
    train_rows = all_train[:TRAIN_REVIEW_N]
    for i, row in enumerate(train_rows, start=1):
        row["priority"] = i
    for row in train_rows:
        cv, hint = row["_cv"], row["search_hint"]
        for s, e, lbl in row["_fn"]:
            train_detail.append({"split": "train", "idx": row["idx"], "type": "FN", "label": lbl,
                                 "snippet": span_snippet(cv["text"], s, e), "search_hint": hint})
        for s, e, lbl in row["_fp"]:
            train_detail.append({"split": "train", "idx": row["idx"], "type": "FP", "label": lbl,
                                 "snippet": span_snippet(cv["text"], s, e), "search_hint": hint})


CSV_FIELDS = [
    "priority", "split", "idx", "fingerprint", "search_hint",
    "n_gold", "n_errors", "n_fn", "n_fp", "f1_doc",
    "err_SKILL", "err_INSTITUTION", "err_COMPANY", "err_DEGREE", "err_JOB_TITLE", "err_DATE",
]


def write_csv(path, rows):
    with open(path, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=CSV_FIELDS, extrasaction="ignore")
        w.writeheader()
        w.writerows(rows)


def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            cv = row["_cv"]
            out = {
                "priority": row["priority"],
                "split": row["split"],
                "idx": row["idx"],
                "fingerprint": row["fingerprint"],
                "search_hint": row["search_hint"],
                "n_errors": row["n_errors"],
                "f1_doc": row["f1_doc"],
                "text": cv["text"],
                "gold_entities": [
                    {"start": s, "end": e, "label": l}
                    for s, e, l in cv.get("entities", [])
                ],
            }
            f.write(json.dumps(out, ensure_ascii=False) + "\n")


dev_csv = os.path.join(AUDIT_DIR, "doccano_review_60dev.csv")
train_csv = os.path.join(AUDIT_DIR, "doccano_review_20train.csv")
dev_jsonl = os.path.join(AUDIT_DIR, "doccano_review_60dev.jsonl")
train_jsonl = os.path.join(AUDIT_DIR, "doccano_review_20train.jsonl")
detail_jsonl = os.path.join(AUDIT_DIR, "errors_detail.jsonl")

write_csv(dev_csv, dev_rows)
write_jsonl(dev_jsonl, dev_rows)
write_csv(os.path.join(AUDIT_DIR, "dev_errors_ranked.csv"), dev_rows)

if train_rows:
    write_csv(train_csv, train_rows)
    write_jsonl(train_jsonl, train_rows)
    write_csv(os.path.join(AUDIT_DIR, "train_to_review.csv"), train_rows)

with open(detail_jsonl, "w", encoding="utf-8") as f:
    for item in dev_detail + train_detail:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

checklist = os.path.join(AUDIT_DIR, "DOCANNO_CHECKLIST.txt")
with open(checklist, "w", encoding="utf-8") as f:
    f.write("DOCANNO — 60 DEV + 20 TRAIN (6 labels NER)\n")
    f.write("=" * 55 + "\n\nDEV — priority #1 = plus d'erreurs\n")
    for row in dev_rows:
        f.write(f"#{row['priority']:2d} | err={row['n_errors']:2d} | {row['search_hint'][:100]}\n")
    f.write("\nTRAIN — top 20 erreurs\n")
    for row in train_rows:
        f.write(f"#{row['priority']:2d} | err={row['n_errors']:2d} | {row['search_hint'][:100]}\n")
    f.write("\nRègles: SKILL court | INSTITUTION ≠ DEGREE | pas CERTIFICATION | pas EXPERIENCE_DESC\n")

print("\n" + "=" * 60)
print(f"Modèle : {model_path}")
print(f"60 DEV  → {dev_csv}")
print(f"20 TRAIN → {train_csv if train_rows else 'N/A (train_cv.json manquant)'}")
print(f"Détail  → {detail_jsonl}")
print(f"Liste   → {checklist}")
print(f"Erreurs dev : {dict(dev_label_errors.most_common())}")
print("\nTop 10 dev :")
for row in dev_rows[:10]:
    print(f"  #{row['priority']:2d} err={row['n_errors']:2d} | {row['search_hint'][:70]}...")
print("\nDoccano : Ctrl+F search_hint → corriger → export JSONL → notebook 03")


In [ ]:
#@title 📥 Export JSONL Doccano (labels colorés) — 60 dev + 20 train

import csv
import json
import os

if "BASE" not in globals():
    BASE = "/content/drive/MyDrive/cvextraction/Datasets"
BLOC3 = os.path.join(BASE, "bloc3_processed")
AUDIT_DIR = os.path.join(BLOC3, "annotation_audit")

LABELS_6 = [
    "SKILL", "JOB_TITLE", "COMPANY", "DATE", "DEGREE", "INSTITUTION",
]


def cv_to_doccano_line(cv, meta=None):
    labels = []
    for start, end, label in cv.get("entities", []):
        if label not in LABELS_6:
            continue
        labels.append([int(start), int(end), str(label)])
    labels.sort(key=lambda x: x[0])
    row = {"text": cv["text"], "label": labels}
    if meta:
        row["meta"] = meta
    return row


dev_cv_path = os.path.join(BLOC3, "dev_cv.json")
dev_out = os.path.join(AUDIT_DIR, "doccano_import_60dev.jsonl")

with open(dev_cv_path, encoding="utf-8") as f:
    dev_all = json.load(f)

priority_csv = os.path.join(AUDIT_DIR, "doccano_review_60dev.csv")
if os.path.isfile(priority_csv):
    fps = []
    with open(priority_csv, encoding="utf-8") as f:
        for row in csv.DictReader(f):
            fps.append(row["fingerprint"])
    by_fp = {c.get("fingerprint"): c for c in dev_all}
    dev_ordered = [by_fp[fp] for fp in fps if fp in by_fp]
else:
    dev_ordered = dev_all

with open(dev_out, "w", encoding="utf-8") as out:
    for i, cv in enumerate(dev_ordered):
        line = cv_to_doccano_line(cv, {"priority": i + 1, "split": "dev"})
        out.write(json.dumps(line, ensure_ascii=False) + "\n")

print(f"60 DEV → {dev_out} ({len(dev_ordered)} docs)")

train_out = os.path.join(AUDIT_DIR, "doccano_import_20train.jsonl")
train_csv = os.path.join(AUDIT_DIR, "doccano_review_20train.csv")
train_cv_path = os.path.join(BLOC3, "train_cv.json")

if os.path.isfile(train_csv) and os.path.isfile(train_cv_path):
    with open(train_cv_path, encoding="utf-8") as f:
        by_fp = {c.get("fingerprint"): c for c in json.load(f)}
    train_ordered = []
    with open(train_csv, encoding="utf-8") as f:
        for row in csv.DictReader(f):
            cv = by_fp.get(row["fingerprint"])
            if cv:
                train_ordered.append(cv)
    with open(train_out, "w", encoding="utf-8") as out:
        for i, cv in enumerate(train_ordered):
            line = cv_to_doccano_line(cv, {"priority": i + 1, "split": "train"})
            out.write(json.dumps(line, ensure_ascii=False) + "\n")
    print(f"20 TRAIN → {train_out} ({len(train_ordered)} docs)")

print("""
IMPORT DOCCANO — labels colorés
===============================
1. Type projet : Sequence Labeling
2. Labels : SKILL, JOB_TITLE, COMPANY, DATE, DEGREE, INSTITUTION
3. Dataset → Import JSONL : doccano_import_60dev.jsonl

OU (plus simple) : projet Doccano ORIGINAL (~400 CV)
→ Ctrl+F search_hint du CSV → corriger sur place
""")

In [ ]:
#@title 🔧 Réparer copie modèle → Drive (si audit échoue)

import os, shutil
from pathlib import Path

if "BASE" not in globals():
    BASE = "/content/drive/MyDrive/cvextraction/Datasets"
MODEL_DIR = os.path.join(BASE, "models")
RUN_DIR = os.path.join(MODEL_DIR, "cv_ner_trf_run_6l")
FINAL = os.path.join(MODEL_DIR, "cv_ner_roberta_best")

for name in ["model-best", "model-last"]:
    src = os.path.join(RUN_DIR, name)
    tr = Path(src) / "transformer" / "model"
    if tr.exists():
        mb = tr.stat().st_size // 1024 // 1024
        print(f"Source OK : {src} (transformer/model ~{mb} Mo)")
        if os.path.isdir(FINAL):
            shutil.rmtree(FINAL)
        shutil.copytree(src, FINAL)
        print(f"Copié vers : {FINAL}")
        break
else:
    print("ERREUR : transformer/model introuvable dans", RUN_DIR)
    if Path(RUN_DIR).exists():
        for p in Path(RUN_DIR).rglob("transformer/*"):
            print(" ", p)

In [ ]:
#@title 📊 Évaluation 6 labels

import json
from spacy.scorer import Scorer
from spacy.training import Example

nlp_eval = spacy.load(FINAL_MODEL)
examples = [Example(nlp_eval(d.text), d) for d in dev_docs]
scores = Scorer(nlp_eval).score(examples)
f1 = scores.get("ents_f", 0)
print(f"F1 global (6 labels) : {f1*100:.2f}% | cible {TARGET_F1*100:.0f}%")
per = scores.get("ents_per_type", {})
for label in LABELS:
    m = per.get(label, {})
    print(f"  {label:20s} F1={m.get('f',0)*100:.1f}%")


/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/thinc/shims/pytorch.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self._mixed_precision):
/usr/local/lib/python3.12/dist-packages/thinc/shims/pytorch.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self._mixed_precision):


F1 global (6 labels) : 75.68% | cible 85%
  SKILL                F1=73.6%
  JOB_TITLE            F1=81.4%
  COMPANY              F1=68.8%
  DATE                 F1=90.2%
  DEGREE               F1=86.0%
  INSTITUTION          F1=68.6%
